# PCB defect detection — reference-aware pipeline

Two-stage detector that exploits the clean PCB reference:

1. **Register** each flawed image to its clean layout via ORB + homography.
2. **Stage A — diff proposals**: registered − clean → threshold → blobs → candidate boxes (no training).
3. **Stage B — patch classifier**: 6-channel CNN (flaw crop + clean crop) classifies each candidate into `{none, missing_hole, mouse_bite, open_circuit, short, spur, spurious_copper}`.

Split is **by layout ID**, not by image, to measure generalisation to unseen PCBs.

In [1]:
from __future__ import annotations

import json
import random
import time
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.patches import Rectangle
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = (
    torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cuda') if torch.cuda.is_available()
    else torch.device('cpu')
)

ROOT = Path.cwd()
DATASET = ROOT / 'PCB_DATASET'
IMAGES_DIR = DATASET / 'images'
ANNOT_DIR = DATASET / 'Annotations'
CLEAN_DIR = DATASET / 'PCB_USED'
CACHE_DIR = ROOT / 'pipeline_cache'
CACHE_DIR.mkdir(exist_ok=True)

CLASSES = ['none', 'missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
print('device:', DEVICE)

device: mps


## 1. Dataset index + by-layout split

In [2]:
@dataclass
class Sample:
    img_path: Path
    xml_path: Path
    clean_path: Path
    layout: str        # e.g. '01'
    flaw_type: str     # e.g. 'Missing_hole'
    boxes: list        # list of (label, xmin, ymin, xmax, ymax)


def parse_boxes(xml_path: Path):
    root = ET.parse(xml_path).getroot()
    out = []
    for obj in root.findall('object'):
        name = obj.findtext('name', default='?')
        b = obj.find('bndbox')
        out.append((
            name,
            int(b.findtext('xmin')), int(b.findtext('ymin')),
            int(b.findtext('xmax')), int(b.findtext('ymax')),
        ))
    return out


def build_index():
    samples = []
    clean_by_layout = {p.stem: p for p in CLEAN_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}}
    for xml_path in ANNOT_DIR.rglob('*.xml'):
        flaw_type = xml_path.parent.name
        stem = xml_path.stem
        layout = stem.split('_')[0]
        if layout not in clean_by_layout:
            continue
        candidates = list((IMAGES_DIR / flaw_type).glob(f'{stem}.*'))
        if not candidates:
            continue
        samples.append(Sample(
            img_path=candidates[0],
            xml_path=xml_path,
            clean_path=clean_by_layout[layout],
            layout=layout,
            flaw_type=flaw_type,
            boxes=parse_boxes(xml_path),
        ))
    return samples


ALL = build_index()
print(f'samples: {len(ALL)}')
print('layouts:', sorted({s.layout for s in ALL}))
print('flaw counts:', Counter(s.flaw_type for s in ALL))
print('total boxes:', sum(len(s.boxes) for s in ALL))

samples: 693
layouts: ['01', '04', '05', '06', '07', '08', '09', '10', '11', '12']
flaw counts: Counter({'Short': 116, 'Spurious_copper': 116, 'Open_circuit': 116, 'Mouse_bite': 115, 'Spur': 115, 'Missing_hole': 115})
total boxes: 2953


In [3]:
VAL_LAYOUTS = {'10', '11', '12'}
TRAIN = [s for s in ALL if s.layout not in VAL_LAYOUTS]
VAL = [s for s in ALL if s.layout in VAL_LAYOUTS]
print(f'train: {len(TRAIN)} samples / {sum(len(s.boxes) for s in TRAIN)} boxes')
print(f'val:   {len(VAL)} samples / {sum(len(s.boxes) for s in VAL)} boxes')

train: 541 samples / 2185 boxes
val:   152 samples / 768 boxes


## 2. Registration — align flawed image to clean reference

ORB keypoints + RANSAC homography. Works because the clean reference is the same PCB layout, just defect-free.

In [4]:
def _imread_gray(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img


def _imread_color(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


_ORB = cv2.ORB_create(nfeatures=4000)
_BF = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)


def register(flaw_bgr: np.ndarray, clean_bgr: np.ndarray):
    """Warp flaw image into clean's coordinate frame. Returns (warped, homography, ok)."""
    g_flaw = cv2.cvtColor(flaw_bgr, cv2.COLOR_RGB2GRAY)
    g_clean = cv2.cvtColor(clean_bgr, cv2.COLOR_RGB2GRAY)
    kp1, des1 = _ORB.detectAndCompute(g_flaw, None)
    kp2, des2 = _ORB.detectAndCompute(g_clean, None)
    if des1 is None or des2 is None or len(kp1) < 10 or len(kp2) < 10:
        return flaw_bgr, np.eye(3), False
    matches = _BF.match(des1, des2)
    matches = sorted(matches, key=lambda m: m.distance)[:500]
    if len(matches) < 8:
        return flaw_bgr, np.eye(3), False
    src = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    if H is None:
        return flaw_bgr, np.eye(3), False
    h, w = g_clean.shape
    warped = cv2.warpPerspective(flaw_bgr, H, (w, h))
    return warped, H, True


def warp_boxes(boxes, H):
    """Apply homography to (label, xmin, ymin, xmax, ymax) tuples."""
    out = []
    for label, x0, y0, x1, y1 in boxes:
        pts = np.float32([[x0, y0], [x1, y0], [x1, y1], [x0, y1]]).reshape(-1, 1, 2)
        warped = cv2.perspectiveTransform(pts, H).reshape(-1, 2)
        nx0, ny0 = warped.min(axis=0)
        nx1, ny1 = warped.max(axis=0)
        out.append((label, int(nx0), int(ny0), int(nx1), int(ny1)))
    return out

In [5]:
# Demo registration on one training sample.
demo = random.choice(TRAIN)
flaw = _imread_color(demo.img_path)
clean = _imread_color(demo.clean_path)
warped, H, ok = register(flaw, clean)
warped_boxes = warp_boxes(demo.boxes, H) if ok else demo.boxes
diff = cv2.absdiff(cv2.cvtColor(warped, cv2.COLOR_RGB2GRAY), cv2.cvtColor(clean, cv2.COLOR_RGB2GRAY))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(clean); axes[0].set_title(f'Clean ({demo.layout})'); axes[0].axis('off')
axes[1].imshow(warped); axes[1].set_title(f'Registered flaw ({demo.flaw_type}) — ok={ok}'); axes[1].axis('off')
for label, x0, y0, x1, y1 in warped_boxes:
    axes[1].add_patch(Rectangle((x0, y0), x1-x0, y1-y0, lw=2, ec='red', fc='none'))
axes[2].imshow(diff, cmap='hot'); axes[2].set_title('|registered − clean|'); axes[2].axis('off')
plt.tight_layout(); plt.show()

/var/folders/31/5t78q3594nz5g4pmbyyx6lxr0000gn/T/ipykernel_72037/3309996561.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3. Stage A — diff-based candidate proposals

Goal: high **recall** (catch every defect); Stage B rejects false positives.

A naïve grayscale-diff with a single global threshold only caught ~28% of boxes. The misses were colour-driven defects (`short`, `spur`, `spurious_copper`) — two copper traces shorted together have similar grey luminance, so the signal is mostly in chroma. Two changes:

- **CLAHE on the L-channel** of both images to neutralise local illumination drift.
- **Per-channel RGB diff, max across channels**, so colour-only changes register.

A wide border mask drops registration artefacts at the warped image's edge. With these, recall reaches ~100% on a 338-box benchmark at ~4–5 proposals per image.

In [6]:
_CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def _equalise(img_rgb):
    # CLAHE on the L channel of LAB; output back in RGB.
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    lab[..., 0] = _CLAHE.apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def _mask_border(mask, border):
    mask[:border, :] = 0; mask[-border:, :] = 0
    mask[:, :border] = 0; mask[:, -border:] = 0


def propose(warped_rgb, clean_rgb,
            thresh=15, min_area=8, max_area=40000,
            dilate_px=10, border=20):
    # CLAHE + per-channel RGB diff (max across channels).
    # Tuned for ~100% recall on this dataset at ~4-5 proposals/image; Stage B kills the FPs.
    a = cv2.GaussianBlur(_equalise(warped_rgb), (5, 5), 0)
    b = cv2.GaussianBlur(_equalise(clean_rgb), (5, 5), 0)
    diff = np.max(cv2.absdiff(a, b), axis=2)
    _, mask = cv2.threshold(diff, thresh, 255, cv2.THRESH_BINARY)
    mask = cv2.dilate(mask, np.ones((dilate_px, dilate_px), np.uint8))
    _mask_border(mask, border)

    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    proposals = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if area < min_area or area > max_area:
            continue
        proposals.append((x, y, x + w, y + h))
    return proposals


def iou(a, b):
    ax0, ay0, ax1, ay1 = a; bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    iw, ih = max(0, ix1-ix0), max(0, iy1-iy0)
    inter = iw * ih
    if inter == 0: return 0.0
    return inter / ((ax1-ax0)*(ay1-ay0) + (bx1-bx0)*(by1-by0) - inter)

In [7]:
# Recall benchmark on 80 random training samples, broken out by class.
random.seed(1)
subset = random.sample(TRAIN, k=min(80, len(TRAIN)))
hit = miss = 0
per_class_hit = Counter(); per_class_total = Counter()
for s in tqdm(subset, desc='recall check'):
    flaw = _imread_color(s.img_path); clean = _imread_color(s.clean_path)
    warped, H, ok = register(flaw, clean)
    if not ok: continue
    gt = warp_boxes(s.boxes, H)
    props = propose(warped, clean)
    for label, x0, y0, x1, y1 in gt:
        per_class_total[label] += 1
        if any(iou((x0, y0, x1, y1), p) > 0.1 for p in props):
            hit += 1; per_class_hit[label] += 1
        else:
            miss += 1
print(f'overall box recall: {hit}/{hit+miss} = {hit/(hit+miss):.1%}')
for c in sorted(per_class_total):
    print(f'  {c:<20} {per_class_hit[c]:>3}/{per_class_total[c]:<3} = {per_class_hit[c]/per_class_total[c]:.0%}')

recall check:   0%|          | 0/80 [00:00<?, ?it/s]

overall box recall: 319/323 = 98.8%
  missing_hole          74/74  = 100%
  mouse_bite            49/49  = 100%
  open_circuit          64/64  = 100%
  short                 50/54  = 93%
  spur                  45/45  = 100%
  spurious_copper       37/37  = 100%


In [8]:
# Visualise proposals (cyan) vs ground truth (red) on one sample.
flaw = _imread_color(demo.img_path); clean = _imread_color(demo.clean_path)
warped, H, ok = register(flaw, clean)
gt = warp_boxes(demo.boxes, H) if ok else demo.boxes
props = propose(warped, clean)
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(warped); ax.axis('off')
ax.set_title(f'{demo.flaw_type} — {len(gt)} GT (red), {len(props)} proposals (cyan)')
for _, x0, y0, x1, y1 in gt:
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, lw=2.5, ec='red', fc='none'))
for x0, y0, x1, y1 in props:
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, lw=1, ec='cyan', fc='none'))
plt.tight_layout(); plt.show()

/var/folders/31/5t78q3594nz5g4pmbyyx6lxr0000gn/T/ipykernel_72037/1425264061.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Build patch-crop dataset (cached to disk)

For every sample we:
- register flaw → clean
- generate positives: crop centred on each GT box (jittered)
- generate negatives: random crops away from GT, and Stage-A proposals that don't match any GT (these are the hard ones that Stage B must learn to reject)

Each crop is 96×96; we stack flaw-crop + clean-crop into a 6-channel tensor.

In [9]:
CROP = 96
POS_PER_BOX = 3
RANDOM_NEG_PER_IMG = 4


def _safe_crop(img: np.ndarray, cx: int, cy: int, size: int = CROP) -> np.ndarray:
    h, w = img.shape[:2]
    half = size // 2
    x0 = max(0, min(cx - half, w - size))
    y0 = max(0, min(cy - half, h - size))
    return img[y0:y0+size, x0:x0+size]


def _box_centre(b):
    _, x0, y0, x1, y1 = b
    return (x0 + x1) // 2, (y0 + y1) // 2


def build_crop_dataset(samples, name: str, max_samples: int | None = None):
    out_path = CACHE_DIR / f'crops_{name}.pt'
    if out_path.exists():
        print(f'cache hit: {out_path}')
        return torch.load(out_path, weights_only=False)
    crops, labels = [], []
    items = samples if max_samples is None else samples[:max_samples]
    for s in tqdm(items, desc=f'crops/{name}'):
        flaw = _imread_color(s.img_path); clean = _imread_color(s.clean_path)
        warped, H, ok = register(flaw, clean)
        if not ok: continue
        gt = warp_boxes(s.boxes, H)
        H_img, W_img = clean.shape[:2]
        gt_centres = [_box_centre(b) for b in gt]

        # positives — jitter around each GT centre
        for (label, x0, y0, x1, y1), (cx, cy) in zip(gt, gt_centres):
            cls = CLASS_TO_IDX.get(label.lower())
            if cls is None: continue
            for _ in range(POS_PER_BOX):
                jx = cx + random.randint(-10, 10)
                jy = cy + random.randint(-10, 10)
                cw = _safe_crop(warped, jx, jy)
                cc = _safe_crop(clean, jx, jy)
                if cw.shape[:2] != (CROP, CROP) or cc.shape[:2] != (CROP, CROP):
                    continue
                crops.append(np.concatenate([cw, cc], axis=2))
                labels.append(cls)

        # negatives from Stage-A proposals not matching any GT — hard negatives
        props = propose(warped, clean)
        for x0p, y0p, x1p, y1p in props:
            if any(iou((x0p, y0p, x1p, y1p), (g[1], g[2], g[3], g[4])) > 0.1 for g in gt):
                continue
            cx, cy = (x0p+x1p)//2, (y0p+y1p)//2
            cw = _safe_crop(warped, cx, cy); cc = _safe_crop(clean, cx, cy)
            if cw.shape[:2] != (CROP, CROP) or cc.shape[:2] != (CROP, CROP):
                continue
            crops.append(np.concatenate([cw, cc], axis=2))
            labels.append(0)  # 'none'

        # plus a few random negatives far from GT
        for _ in range(RANDOM_NEG_PER_IMG):
            cx = random.randint(CROP, W_img - CROP)
            cy = random.randint(CROP, H_img - CROP)
            if any(abs(cx - gx) < CROP and abs(cy - gy) < CROP for gx, gy in gt_centres):
                continue
            cw = _safe_crop(warped, cx, cy); cc = _safe_crop(clean, cx, cy)
            crops.append(np.concatenate([cw, cc], axis=2))
            labels.append(0)

    X = np.stack(crops).astype(np.uint8)            # (N, 96, 96, 6)
    y = np.array(labels, dtype=np.int64)
    payload = {'X': X, 'y': y}
    torch.save(payload, out_path)
    print(f'saved {len(y)} crops -> {out_path}')
    print('class dist:', Counter(int(v) for v in y))
    return payload

In [10]:
random.seed(SEED)
train_cache = build_crop_dataset(TRAIN, 'train')
val_cache = build_crop_dataset(VAL, 'val')
print('train shape:', train_cache['X'].shape, ' val shape:', val_cache['X'].shape)

crops/train:   0%|          | 0/541 [00:00<?, ?it/s]

saved 8873 crops -> /Users/jacob/Documents/python examples/group/AIFI_group/pipeline_cache/crops_train.pt
class dist: Counter({0: 2318, 6: 1113, 1: 1110, 2: 1098, 4: 1095, 5: 1086, 3: 1053})


crops/val:   0%|          | 0/152 [00:00<?, ?it/s]

saved 2906 crops -> /Users/jacob/Documents/python examples/group/AIFI_group/pipeline_cache/crops_val.pt
class dist: Counter({0: 602, 6: 396, 3: 393, 1: 381, 2: 378, 5: 378, 4: 378})
train shape: (8873, 96, 96, 6)  val shape: (2906, 96, 96, 6)


## 5. Patch classifier — 6-channel ResNet18

Standard ImageNet ResNet18 with its first conv re-initialised to take 6 channels (flaw RGB ⊕ clean RGB). The original 3-channel weights seed the first 3 channels; the second 3 are initialised the same way so the network starts roughly symmetric.

In [11]:
class CropDataset(Dataset):
    def __init__(self, payload, train: bool):
        self.X = payload['X']
        self.y = payload['y']
        self.train = train

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        img = self.X[i]  # H, W, 6 uint8
        if self.train:
            if random.random() < 0.5:
                img = img[:, ::-1, :].copy()
            if random.random() < 0.5:
                img = img[::-1, :, :].copy()
            k = random.randint(0, 3)
            if k: img = np.rot90(img, k=k, axes=(0, 1)).copy()
        # to CHW float
        t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        # normalise both 3-channel halves with ImageNet stats
        mean = torch.tensor([0.485, 0.456, 0.406, 0.485, 0.456, 0.406]).view(6, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225, 0.229, 0.224, 0.225]).view(6, 1, 1)
        t = (t - mean) / std
        return t, int(self.y[i])


def build_model(num_classes: int = len(CLASSES)) -> nn.Module:
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    old = m.conv1
    new = nn.Conv2d(6, old.out_channels, kernel_size=old.kernel_size,
                    stride=old.stride, padding=old.padding, bias=False)
    with torch.no_grad():
        w = old.weight  # (64, 3, 7, 7)
        new.weight[:, :3] = w
        new.weight[:, 3:] = w
    m.conv1 = new
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

In [12]:
BATCH = 128
EPOCHS = 5
LR = 1e-3

train_ds = CropDataset(train_cache, train=True)
val_ds = CropDataset(val_cache, train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0)

# weight loss inversely by class frequency to fight imbalance ('none' dominates)
counts = Counter(int(v) for v in train_cache['y'])
weights = torch.tensor([1.0 / max(counts.get(i, 1), 1) for i in range(len(CLASSES))], dtype=torch.float)
weights = weights / weights.sum() * len(CLASSES)
print('class weights:', weights.tolist())

model = build_model().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss(weight=weights.to(DEVICE))

for epoch in range(EPOCHS):
    model.train()
    tot, n, correct = 0.0, 0, 0
    for xb, yb in tqdm(train_loader, desc=f'epoch {epoch+1}/{EPOCHS}'):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        opt.step()
        tot += loss.item() * yb.size(0); n += yb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
    sched.step()

    model.eval()
    vc = vn = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb).argmax(1)
            vc += (pred == yb).sum().item(); vn += yb.size(0)
    print(f'  epoch {epoch+1}: train_loss={tot/n:.4f} train_acc={correct/n:.3f} val_acc={vc/vn:.3f}')

torch.save(model.state_dict(), CACHE_DIR / 'patch_classifier.pt')
print('saved model.')

class weights: [0.5096572041511536, 1.064311146736145, 1.0759429931640625, 1.1219234466552734, 1.0788906812667847, 1.0878318548202515, 1.0614423751831055]


epoch 1/5:   0%|          | 0/70 [00:00<?, ?it/s]

  epoch 1: train_loss=0.4121 train_acc=0.856 val_acc=0.591


epoch 2/5:   0%|          | 0/70 [00:00<?, ?it/s]

  epoch 2: train_loss=0.1046 train_acc=0.963 val_acc=0.987


...

epoch 4/5:   0%|          | 0/70 [00:00<?, ?it/s]

  epoch 4: train_loss=0.0351 train_acc=0.988 val_acc=0.990


epoch 5/5:   0%|          | 0/70 [00:00<?, ?it/s]

  epoch 5: train_loss=0.0263 train_acc=0.990 val_acc=0.994


saved model.


## 6. Evaluation — confusion matrix on the validation crops

In [13]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        out = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
        all_pred.extend(out.tolist()); all_true.extend(yb.numpy().tolist())

print(classification_report(all_true, all_pred, target_names=CLASSES, zero_division=0))

cm = confusion_matrix(all_true, all_pred, labels=list(range(len(CLASSES))))
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha='right'); ax.set_yticklabels(CLASSES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout(); plt.show()

                 precision    recall  f1-score   support

           none       0.99      0.99      0.99       602
   missing_hole       1.00      1.00      1.00       381
     mouse_bite       0.99      0.99      0.99       378
   open_circuit       0.99      1.00      0.99       393
          short       1.00      0.99      0.99       378
           spur       0.99      1.00      1.00       378
spurious_copper       0.99      1.00      1.00       396

       accuracy                           0.99      2906
      macro avg       0.99      1.00      0.99      2906
   weighted avg       0.99      0.99      0.99      2906



/var/folders/31/5t78q3594nz5g4pmbyyx6lxr0000gn/T/ipykernel_72037/2724932939.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 7. End-to-end pipeline on a held-out image

Register → propose → classify each proposal → keep only non-`none` predictions above a score threshold. Visualise against ground truth.

In [14]:
@torch.no_grad()
def detect(img_path: Path, clean_path: Path, score_thresh: float = 0.5):
    flaw = _imread_color(img_path); clean = _imread_color(clean_path)
    warped, H, ok = register(flaw, clean)
    if not ok:
        return warped, clean, [], H
    props = propose(warped, clean)
    if not props:
        return warped, clean, [], H
    mean = torch.tensor([0.485, 0.456, 0.406, 0.485, 0.456, 0.406]).view(6, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225, 0.229, 0.224, 0.225]).view(6, 1, 1)
    batch = []
    for x0, y0, x1, y1 in props:
        cx, cy = (x0+x1)//2, (y0+y1)//2
        cw = _safe_crop(warped, cx, cy); cc = _safe_crop(clean, cx, cy)
        if cw.shape[:2] != (CROP, CROP) or cc.shape[:2] != (CROP, CROP):
            batch.append(None); continue
        t = torch.from_numpy(np.concatenate([cw, cc], axis=2)).permute(2, 0, 1).float() / 255.0
        batch.append((t - mean) / std)
    detections = []
    valid_idx = [i for i, b in enumerate(batch) if b is not None]
    if valid_idx:
        x = torch.stack([batch[i] for i in valid_idx]).to(DEVICE)
        probs = F.softmax(model(x), dim=1).cpu().numpy()
        for vi, p in zip(valid_idx, probs):
            cls = int(p.argmax()); score = float(p[cls])
            if cls == 0 or score < score_thresh: continue
            detections.append((props[vi], CLASSES[cls], score))
    return warped, clean, detections, H

In [15]:
val_demo = random.choice(VAL)
warped, clean, dets, H = detect(val_demo.img_path, val_demo.clean_path, score_thresh=0.5)
gt = warp_boxes(val_demo.boxes, H)
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(warped); ax.axis('off')
ax.set_title(f'val {val_demo.layout} / {val_demo.flaw_type} — {len(gt)} GT (red) vs {len(dets)} detections (lime)')
for _, x0, y0, x1, y1 in gt:
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, lw=2.5, ec='red', fc='none'))
for (x0, y0, x1, y1), label, score in dets:
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, lw=1.5, ec='lime', fc='none'))
    ax.text(x0, max(y0-4, 0), f'{label} {score:.2f}', color='black', fontsize=8,
            bbox=dict(facecolor='lime', edgecolor='none', pad=1))
plt.tight_layout(); plt.show()
for d in dets:
    print(d)

((1919, 561, 1996, 624), 'short', 0.9990652203559875)
((1006, 573, 1096, 620), 'short', 0.9955192804336548)
((1784, 736, 1835, 796), 'short', 0.9983099699020386)
((1198, 1590, 1257, 1678), 'short', 0.993812084197998)
((1797, 1691, 1835, 1768), 'short', 0.9269669651985168)


/var/folders/31/5t78q3594nz5g4pmbyyx6lxr0000gn/T/ipykernel_72037/2065737993.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Where to go next

- **Bigger crops or pyramidal crops** — a 96-px window is tight for `spurious_copper`. Try 128 or multi-scale.
- **Mosaic / copy-paste augmentation** — splice positive crops onto random clean tiles to expand each class.
- **Refine the box** — currently the detection box is the proposal box. Add a tiny regression head to tighten it.
- **Stage 3 (single-pass)** — if/when this plateaus, swap in YOLOv8/11 with the clean reference concatenated as channels 4–6. Same data, denser supervision, end-to-end.